In [2]:
import py_vncorenlp
import os
import nltk

In [3]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-26" 
try:
    path # type: ignore
except:
    path=os.getcwd()
    model = py_vncorenlp.VnCoreNLP(save_dir=f"{path}\\VnCoreNLP", annotators=["wseg", "pos"]) # type: ignore
    os.chdir("..")

In [4]:
raw_texts= []
with open(f"{path}/output/clauses.txt", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        raw_texts.append(line.strip())
raw_texts

['Bên A cung cấp dịch vụ phát triển Hệ thống Quản lý Bệnh viện Điện tử ( HIS ) phiên bản 3.0 .',
 'Phạm vi : phân tích , thiết kế , phát triển , triển khai , đào tạo và bảo trì 24 tháng .',
 'Bên A cam kết hệ thống đáp ứng tiêu chuẩn bảo mật dữ liệu y tế theo Bộ Y tế .',
 'Bên A báo cáo tiến độ mỗi 02 tuần và họp review hàng tháng .',
 'Chậm tiến độ quá 15 ngày , Bên B có quyền yêu cầu tăng nhân lực hoặc phạt theo Điều 7 .',
 'Bên A không được sử dụng mã nguồn tuỳ chỉnh cho bên thứ ba .',
 'Bên B có quyền yêu cầu chuyển giao toàn bộ tài liệu kỹ thuật .',
 'Bên A cam kết bảo mật dữ liệu bệnh nhân , hồ sơ y tế và thông tin kinh doanh .',
 'Nghiêm cấm Bên A truy cập , sao chép hoặc chia sẻ dữ liệu bệnh nhân cho bên thứ ba .',
 'Nghĩa vụ bảo mật kéo dài vô thời hạn đối với dữ liệu y tế .',
 'Vi phạm bảo mật : bồi thường tối thiểu 500.000.000 VNĐ .',
 'Bảo hành 24 tháng kể từ ngày nghiệm thu .',
 'Hỗ trợ kỹ thuật 24/7 trong thời gian bảo hành .',
 'Chậm tiến độ : phạt 0.3% giá trị hợp đồng 

In [12]:
import nltk
import csv
import os

grammar = r"""
    NP: {<N><Np|Ny|M>}                                      
        {<L>?<M>?<Nc>?<N|Np|Nu|Ny>+(<A|V|N|Np|Ny>)*<P>?}    
"""
chunk_parser = nltk.RegexpParser(grammar)

# Chuẩn bị đường dẫn file output
txt_output_file = f"{path}/output/chunks.txt"
csv_output_file = f"{path}/output/chunks.csv"

# Đảm bảo thư mục output tồn tại để không bị lỗi
os.makedirs(f"{path}/output", exist_ok=True)

# Khởi tạo ID câu (Rất cần thiết cho file CSV chuẩn NER)
sentence_id = 0 

# Mở đồng thời cả file TXT và file CSV
with open(txt_output_file, "w", encoding="utf-8") as f_txt, \
     open(csv_output_file, "w", encoding="utf-8", newline='') as f_csv:
    
    # Khởi tạo trình ghi CSV và ghi dòng Tiêu đề (Header)
    csv_writer = csv.writer(f_csv)
    csv_writer.writerow(["Sentence_ID", "Word", "POS", "Tag"])

    for line in raw_texts:
        annotated_sentences = model.annotate_text(line)
        
        for sentence_index, tokens in annotated_sentences.items():
            tagged_tokens = [(token["wordForm"], token["posTag"]) for token in tokens]
            
            tree = chunk_parser.parse(tagged_tokens)
            
            bio_tags = []
            for subtree in tree:
                if type(subtree) == nltk.Tree and subtree.label() == 'NP':
                    for i, (word, pos) in enumerate(subtree.leaves()):
                        tag = 'B-NP' if i == 0 else 'I-NP'
                        bio_tags.append((word, pos, tag))
                else:
                    word, pos = subtree
                    bio_tags.append((word, pos, 'O'))
            
            for word, pos, tag in bio_tags:
                # Xử lý gán nhãn cho các âm tiết bị gạch dưới (VD: phát_triển)
                next_tag = "I" + tag[1:] if tag != "O" else "O"
                word_list = word.split("_")
                
                tag_list = [tag] + [next_tag] * (len(word_list) - 1)
                pos_list = [pos] * len(word_list) 
                
                for syllable, p, t in zip(word_list, pos_list, tag_list):
                    # 1. Ghi vào file TXT (chuẩn CoNLL)
                    f_txt.write(f"{syllable:<15} {t}\n")
                    
                    # 2. Ghi vào file CSV
                    csv_writer.writerow([sentence_id, syllable, p, t])
            
            # Kết thúc một câu: Xuống dòng trong file TXT và Tăng ID câu
            f_txt.write("\n")
            sentence_id += 1

print("✅ Đã xử lý xong. Dữ liệu được lưu tại 2 định dạng:")
print(f"📄 Chuẩn CoNLL (TXT): {txt_output_file}")
print(f"📊 Chuẩn Dataset (CSV): {csv_output_file}")

✅ Đã xử lý xong. Dữ liệu được lưu tại 2 định dạng:
📄 Chuẩn CoNLL (TXT): c:\Users\thien\OneDrive\Desktop\Ass_NLP/output/chunks.txt
📊 Chuẩn Dataset (CSV): c:\Users\thien\OneDrive\Desktop\Ass_NLP/output/chunks.csv
